In [28]:
import time
import copy
import re
from types import SimpleNamespace
from IPython.display import clear_output

# ── Colori ────────────────────────────────────────────────────────────────────
ANSI_COLORS = ["\033[91m", "\033[92m", "\033[93m", "\033[94m", "\033[95m", "\033[96m"]
YELLOW_STATION = "\033[1;93m"
RESET = "\033[0m"

def init_key_colors(stanze):
    keys = [item for item in stanze if item.startswith("key")]
    return {key: ANSI_COLORS[i % len(ANSI_COLORS)] for i, key in enumerate(keys)}

def colored_key(key_name, key_colors):
    color = key_colors.get(key_name, "")
    return f"{color}k{RESET}"

def door_color(door_name, key_colors):
    key_name = door_name.replace("door_", "key_")
    return key_colors.get(key_name, "")

# ── Funzione di stampa  ───────────────────────────────
def draw_grid(azione_corrente, stanze, bal, batteria, porte, mani, key_colors):
    clear_output(wait=True)
    print(porte)
    print()
    print("Batteria: ", batteria)
    print("Azione eseguita: ", azione_corrente)
    print()

    appStanze = len(stanze)
    appBal    = bal
    c         = RESET

    for k in range(2):
        if appStanze % 2 == 1:
            print("      ", end="")
        for j in range(appStanze // 2):
            idx = abs(k * (len(stanze) - 2) - j) + k
            if (j + 1) == appStanze // 2 and k == 1:
                idx -= 1
            a = idx
            b = (idx + 1) % len(stanze)
            if idx == 3 or (k == 1 and b == 3):
                c = YELLOW_STATION
            else:
                c = RESET
            if len(stanze) % 2 == 0:
                print(f"{c}+-", end="")
                if k == 1 and (j == 0 or j == appStanze // 2 - 1):
                    print(f"{c}---{RESET}", end="")
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}  |{RESET}", end="")
                    else:
                        print(f"{color}___{RESET}", end="")
                    print(f"{c}---", end="")
                else:
                    print(f"{c}---------", end="")
                print(f"{c}-", end="")
            else:
                print(f"{c}+", end="")
                if k == 1 and j == appStanze // 2 - 1:
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}    |{RESET}", end="")
                    else:
                        print(f"{color}_____{RESET}", end="")
                else:
                    print(f"{c}-----", end="")
                print(f"{c}-{RESET}" if k == 0 else f"{c}+{RESET}", end="")
                if k == 1 and j == 0:
                    color = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                    if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                        print(f"{color}    |{RESET}", end="")
                    else:
                        print(f"{color}_____{RESET}", end="")
                else:
                    print(f"{c}-----", end="")
        print(f"{c}+")

        # parte centrale
        for l in range(5):
            if appStanze % 2 == 1:
                print("      ", end="")
            for j in range(appStanze // 2):
                idx = abs(k * (len(stanze) - 2) - j) + k + k - 1
                a = idx
                b = (idx + 1) % len(stanze)
                room_idx = abs(k * (len(stanze) - 1) - j)
                room_content = stanze[room_idx]

                if a == 3 or b == 3:
                    c = YELLOW_STATION
                else:
                    c = RESET

                if l != 2 or j == 0:
                    print(f"{c}|{RESET}", end="")
                else:
                  dcolor = door_color(f"door_{min(a,b)}_{max(a,b)}", key_colors)
                  if porte[f"door_{min(a,b)}_{max(a,b)}"] == "open":
                    print(f"{dcolor}_{RESET}", end="")
                  else:
                      print(f"{dcolor}I{RESET}", end="")
                if bal == room_idx:
                    if l == 1:
                        print("    (O)    ", end="")
                    elif l == 2:
                        l_hand = colored_key(mani[0], key_colors) if mani[0].startswith("key") else mani[0]
                        r_hand = colored_key(mani[1], key_colors) if mani[1].startswith("key") else mani[1]
                        print(f"  {l_hand}/|#|\\{r_hand}  ", end="")
                    elif l == 3:
                        print("    / \\    ", end="")
                    elif room_content.startswith("box") and l == 0:
                        print(f"    [{room_content[3:]}]    ", end="")
                    elif room_content.startswith("key") and l == 0:
                        print(f"     {colored_key(room_content, key_colors)}     ", end="")
                    else:
                        print("           ", end="")
                elif room_content.startswith("box") and l == 2:
                    print(f"    [{room_content[3:]}]    ", end="")
                elif room_content.startswith("key") and l == 2:
                    print(f"     {colored_key(room_content, key_colors)}     ", end="")
                else:
                    print("           ", end="")
            if b == 3:
                c = RESET
            print(f"{c}|")

        if k == 1:
            for j in range(appStanze // 2):
                if abs(k * (len(stanze) - 2) - j) + k + k - 1 == 3:
                    c = YELLOW_STATION
                else:
                    c = RESET
                print(f"{c}+-----------", end="")
            print(f"{c}+{RESET}")

        if appStanze % 2 == 1:
            appStanze += 1
            appBal += 1

# ── animate ───────────────────────────────────────────────────────────────────
def animate(sol, initial_state):
    action = sol.solution()
    stato_iniziale = copy.deepcopy(initial_state)
    stanze   = list(stato_iniziale["rooms"])
    bal      = stato_iniziale["robot"]
    batteria = stato_iniziale["battery"]
    porte    = stato_iniziale["doors"]
    mani     = ["o", "o"]

    key_colors = init_key_colors(stanze)

    draw_grid_fn = lambda azione_corrente: draw_grid(
        azione_corrente, stanze, bal, batteria, porte, mani, key_colors
    )

    draw_grid_fn("Stato Iniziale")
    time.sleep(1)

    for az in action:
        parts = az.strip("()").split()
        name  = parts[0]
        batteria -= 1

        if name == "move":
            bal = int(parts[2][4:])

        elif name == "get":
            item = parts[2]
            hand = parts[3]
            stanze[bal] = ""
            hand_item = item[3:] if item.startswith("box") else item   # box1 -> "1"
            if hand == "left":
                mani[0] = hand_item
            else:
                mani[1] = hand_item

        elif name == "put":
            item = parts[2]
            hand = parts[3]
            if hand == "left":
                mani[0] = "o"
            else:
                mani[1] = "o"
            stanze[bal] = item

        elif name == "swap":
            item_in_room = parts[2]
            item_in_hand = parts[3]
            hand         = parts[4]
            hand_item = item_in_room[3:] if item_in_room.startswith("box") else item_in_room
            if hand == "left":
                mani[0] = hand_item
            else:
                mani[1] = hand_item
            stanze[bal] = item_in_hand

        elif name == "open-door":
            porte[parts[3]] = "open"

        elif name == "charge":
            batteria = 10

        draw_grid_fn(az)
        print("\nStanze aggiornate: ", stanze)
        print("Mani aggiornate: ", mani)
        time.sleep(1)


# ── make_problem ──────────────────────────────────────────────────────────────
def make_problem(difficulty: int):
    charging_station = [3]
    max_battery = 10

    configs = {
        1: {
            "robot_start": 0,
            "battery": 10,
            "rooms": ("", "box1", "", ""),
            "doors": {"door_0_1": "open", "door_1_2": "open",
                      "door_2_3": "open", "door_0_3": "open"},
            "goal_robot": 0,
            "goal_rooms": ("box1", "", "", ""),
            "goal_doors": {"door_0_1": "open", "door_1_2": "open",
                           "door_2_3": "open", "door_0_3": "open"},
            "piano_str": (
                "(move room0 room1 door_0_1 b10 b9)"
                "(get room1 box1 left b9 b8)"
                "(move room1 room0 door_0_1 b8 b7)"
                "(put room0 box1 left b7 b6)"
            ),
        },
        2: {
            "robot_start": 0,
            "battery": 10,
            "rooms": ("key_0_1", "", "box1", "", "", ""),
            "doors": {"door_0_1": "closed", "door_1_2": "open", "door_2_3": "open",
                      "door_3_4": "open", "door_4_5": "open", "door_0_5": "open"},
            "goal_robot": 1,
            "goal_rooms": ("", "box1", "", "key_0_1", "", ""),
            "goal_doors": {"door_0_1": "open", "door_1_2": "open", "door_2_3": "open",
                           "door_3_4": "open", "door_4_5": "open", "door_0_5": "open"},
            "piano_str": (
                "(get room0 key_0_1 right b10 b9)"
                "(open-door room0 room1 door_0_1 key_0_1 right b9 b8)"
                "(move room0 room5 door_0_5 b8 b7)"
                "(move room5 room4 door_4_5 b7 b6)"
                "(move room4 room3 door_3_4 b6 b5)"
                "(put room3 key_0_1 right b5 b4)"
                "(move room3 room2 door_2_3 b4 b3)"
                "(get room2 box1 right b3 b2)"
                "(move room2 room1 door_1_2 b2 b1)"
                "(put room1 box1 right b1 b0)"
            ),
        },
        3: {
            "robot_start": 0,
            "battery": 10,
            "rooms": ("key_0_1", "box1", "", "box2", "", "", ""),
            "doors": {"door_0_1": "closed", "door_1_2": "open", "door_2_3": "open",
                      "door_3_4": "open", "door_4_5": "open", "door_5_6": "open",
                      "door_0_6": "open"},
            "goal_robot": 0,
            "goal_rooms": ("", "", "box2", "", "key_0_1", "", "box1"),
            "goal_doors": {"door_0_1": "open", "door_1_2": "open", "door_2_3": "open",
                           "door_3_4": "open", "door_4_5": "open", "door_5_6": "open",
                           "door_0_6": "open"},
            "piano_str": (
                "(get room0 key_0_1 right b10 b9)"
                "(open-door room0 room1 door_0_1 key_0_1 right b9 b8)"
                "(put room0 key_0_1 right b8 b7)"
                "(get room0 key_0_1 right b7 b6)"
                "(move room0 room1 door_0_1 b6 b5)"
                "(move room1 room2 door_1_2 b5 b4)"
                "(move room2 room3 door_2_3 b4 b3)"
                "(move room3 room4 door_3_4 b3 b2)"
                "(put room4 key_0_1 right b2 b1)"
                "(move room4 room3 door_3_4 b1 b0)"
                "(charge room3 b0 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(move room2 room1 door_1_2 b9 b8)"
                "(move room1 room2 door_1_2 b8 b7)"
                "(move room2 room3 door_2_3 b7 b6)"
                "(get room3 box2 right b6 b5)"
                "(move room3 room2 door_2_3 b5 b4)"
                "(put room2 box2 right b4 b3)"
                "(move room2 room3 door_2_3 b3 b2)"
                "(charge room3 b2 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(move room2 room1 door_1_2 b9 b8)"
                "(move room1 room0 door_0_1 b8 b7)"
                "(move room0 room1 door_0_1 b7 b6)"
                "(get room1 box1 left b6 b5)"
                "(move room1 room0 door_0_1 b5 b4)"
                "(move room0 room6 door_0_6 b4 b3)"
                "(put room6 box1 left b3 b2)"
                "(move room6 room0 door_0_6 b2 b1)"
            ),
        },
        4: {
            "robot_start": 7,
            "battery": 10,
            "rooms": ("", "", "key_4_5", "box1", "", "key_1_2", "", ""),
            "doors": {"door_0_1": "open", "door_1_2": "closed", "door_2_3": "open",
                      "door_3_4": "open", "door_4_5": "closed", "door_5_6": "open",
                      "door_6_7": "open", "door_0_7": "open"},
            "goal_robot": 0,
            "goal_rooms": ("", "key_4_5", "box1", "", "", "key_1_2", "", ""),
            "goal_doors": {"door_0_1": "open", "door_1_2": "open", "door_2_3": "open",
                           "door_3_4": "open", "door_4_5": "open", "door_5_6": "open",
                           "door_6_7": "open", "door_0_7": "open"},
            "piano_str": (
                "(move room7 room6 door_6_7 b10 b9)"
                "(move room6 room5 door_5_6 b9 b8)"
                "(get room5 key_1_2 right b8 b7)"
                "(move room5 room6 door_5_6 b7 b6)"
                "(move room6 room7 door_6_7 b6 b5)"
                "(move room7 room0 door_0_7 b5 b4)"
                "(move room0 room1 door_0_1 b4 b3)"
                "(open-door room1 room2 door_1_2 key_1_2 right b3 b2)"
                "(move room1 room2 door_1_2 b2 b1)"
                "(move room2 room3 door_2_3 b1 b0)"
                "(charge room3 b0 b10)"
                "(put room3 key_1_2 right b10 b9)"
                "(move room3 room2 door_2_3 b9 b8)"
                "(get room2 key_4_5 right b8 b7)"
                "(move room2 room1 door_1_2 b7 b6)"
                "(put room1 key_4_5 right b6 b5)"
                "(move room1 room2 door_1_2 b5 b4)"
                "(move room2 room3 door_2_3 b4 b3)"
                "(get room3 box1 right b3 b2)"
                "(get room3 key_1_2 left b2 b1)"
                "(charge room3 b1 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(put room2 box1 right b9 b8)"
                "(put room2 key_1_2 left b8 b7)"
                "(move room2 room1 door_1_2 b7 b6)"
                "(get room1 key_4_5 right b6 b5)"
                "(move room1 room2 door_1_2 b5 b4)"
                "(move room2 room3 door_2_3 b4 b3)"
                "(move room3 room4 door_3_4 b3 b2)"
                "(open-door room4 room5 door_4_5 key_4_5 right b2 b1)"
                "(move room4 room3 door_3_4 b1 b0)"
                "(charge room3 b0 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(put room2 key_4_5 right b9 b8)"
                "(move room2 room3 door_2_3 b8 b7)"
                "(charge room3 b7 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(get room2 key_1_2 right b9 b8)"
                "(move room2 room1 door_1_2 b8 b7)"
                "(move room1 room0 door_0_1 b7 b6)"
                "(move room0 room7 door_0_7 b6 b5)"
                "(move room7 room6 door_6_7 b5 b4)"
                "(move room6 room5 door_5_6 b4 b3)"
                "(put room5 key_1_2 right b3 b2)"
                "(move room5 room4 door_4_5 b2 b1)"
                "(move room4 room3 door_3_4 b1 b0)"
                "(charge room3 b0 b10)"
                "(move room3 room2 door_2_3 b10 b9)"
                "(get room2 key_4_5 right b9 b8)"
                "(move room2 room1 door_1_2 b8 b7)"
                "(put room1 key_4_5 right b7 b6)"
                "(move room1 room0 door_0_1 b6 b5)"
            ),
        },
    }

    assert difficulty in configs, "Difficoltà deve essere 1, 2, 3 o 4"
    c = configs[difficulty]

    initial = {
        "robot": c["robot_start"],
        "left_hand": "",
        "right_hand": "",
        "battery": c["battery"],
        "rooms": c["rooms"],
        "doors": c["doors"],
    }
    goal = {
        "robot": c["goal_robot"],
        "left_hand": "",
        "right_hand": "",
        "rooms": c["goal_rooms"],
        "doors": c["goal_doors"],
    }

    piano = re.findall(r"\([^)]+\)", c["piano_str"])
    sol = SimpleNamespace(
    solution=lambda: piano
    )

    problem = SimpleNamespace(
        initial=initial,
        goal=goal,
        charging_station=charging_station,
        max_battery=max_battery
    )

    return problem, initial, goal, sol


# ── Uso ───────────────────────────────────────────────────────────────────────
DIFFICULTY = 3

problem, initial, goal, sol = make_problem(DIFFICULTY)
animate(sol, initial)

{'door_0_1': 'open', 'door_1_2': 'open', 'door_2_3': 'open', 'door_3_4': 'open', 'door_4_5': 'open', 'door_5_6': 'open', 'door_0_6': 'open'}

Batteria:  1
Azione eseguita:  (move room6 room0 door_0_6 b2 b1)

      +-----------+-----------+-----------+
      |           |           |           |
      |    (O)    |           |           |
      |  o/|#|\o  _           _    [2]    |
      |    / \    |           |           |
      |           |           |           |
+-----+    |+-----+-----+-----+-----+    |+-----+
|           |           |           |           |
|           |           |           |           |
|    [1]    _           _     k     _           |
|           |           |           |           |
|           |           |           |           |
+-----------+-----------+-----------+-----------+

Stanze aggiornate:  ['', '', 'box2', '', 'key_0_1', '', 'box1']
Mani aggiornate:  ['o', 'o']
